In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from scipy.stats import kendalltau

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, DataLoader

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style("whitegrid")


In [ ]:
PATH_9METALS = "HEA_results_9_metals"  
PATH_19METALS = "HEA_results_19_metals" 

ACTIVE_PATH = PATH_19METALS


In [ ]:
def parse_xyz_file(xyz_path):

    with open(xyz_path, 'r') as f:
        lines = f.readlines()
    
    atoms = []
    positions = []
    
    for line in lines[2:]:
        if line.strip() and not line.startswith('!'):
            parts = line.split()
            if len(parts) >= 4:
                atoms.append(parts[0])
                positions.append([float(parts[1]), float(parts[2]), float(parts[3])])
    
    return atoms, np.array(positions)


def find_surface_atoms(positions, z_threshold_percentile=85):
    z_coords = positions[:, 2]
    z_threshold = np.percentile(z_coords, z_threshold_percentile)
    surface_indices = np.where(z_coords >= z_threshold)[0]
    return surface_indices


def compute_neighbor_contributions(atoms, positions, cutoff_distance=3.5):
    surface_indices = find_surface_atoms(positions)
    adsorbate_symbols = ['S', 'H']
    metal_atoms = [atom for atom in atoms if atom not in adsorbate_symbols]
    element_list = sorted(list(set(metal_atoms)))
    element_to_idx = {elem: idx for idx, elem in enumerate(element_list)}
    
    local_features = {}
    
    for surf_idx in surface_indices:
        if atoms[surf_idx] in adsorbate_symbols:
            continue
            
        surf_pos = positions[surf_idx]
        distances = np.linalg.norm(positions - surf_pos, axis=1)
        neighbor_mask = (distances < cutoff_distance) & (distances > 0.1)
        neighbor_indices = np.where(neighbor_mask)[0]
        
        neighbor_composition = np.zeros(len(element_list))
        for neigh_idx in neighbor_indices:
            neigh_elem = atoms[neigh_idx]
            if neigh_elem in element_to_idx:
                neighbor_composition[element_to_idx[neigh_elem]] += 1
        
        total_neighbors = neighbor_composition.sum()
        if total_neighbors > 0:
            neighbor_composition /= total_neighbors
        
        local_features[surf_idx] = neighbor_composition
    
    return local_features, element_list


def aggregate_local_contributions(local_features):
    if not local_features:
        return np.array([])
    
    feature_vectors = list(local_features.values())
    return np.mean(feature_vectors, axis=0)


## 3. Data Loading with Local Neighborhood Features


In [ ]:
def load_composition_data_with_geometry(base_path, composition_folder):
    comp_path = Path(base_path) / composition_folder
    runs_data = []
    
    for run_dir in sorted(comp_path.glob("run_*")):
        json_file = run_dir / "data.json"
        xyz_file = run_dir / "geometry.xyz"
        
        if json_file.exists() and xyz_file.exists():
            with open(json_file, 'r') as f:
                data = json.load(f)
            
            atoms, positions = parse_xyz_file(xyz_file)
            local_features, element_list = compute_neighbor_contributions(atoms, positions)
            
            data['atoms'] = atoms
            data['positions'] = positions
            data['local_features'] = local_features
            data['element_list'] = element_list
            
            runs_data.append(data)
    
    return runs_data


def compute_lq_features(runs_data, composition_name):
    all_E_S = []
    all_E_H = []
    all_local_features = []
    
    for run in runs_data:
        all_E_S.extend(run['energies_S_ads_raw'])
        all_E_H.extend(run['energies_H_ads_raw'])
        
        if 'local_features' in run and run['local_features']:
            aggregated = aggregate_local_contributions(run['local_features'])
            if len(aggregated) > 0:
                all_local_features.append(aggregated)
    
    avg_E_S = np.mean(all_E_S) if all_E_S else 0.0
    avg_E_H = np.mean(all_E_H) if all_E_H else 0.0
    descriptor_D = avg_E_S - avg_E_H
    
    if all_local_features:
        lq_features = np.mean(all_local_features, axis=0)
    else:
        lq_features = parse_composition_string(composition_name)
    
    return lq_features, avg_E_S, avg_E_H, descriptor_D


def parse_composition_string(comp_str):
    elements = ['Ag', 'Au', 'Cd', 'Co', 'Cr', 'Cu', 'Fe', 'Mn', 'Ni', 'Zn', 'Zr']
    composition = {elem: 0 for elem in elements}
    
    i = 0
    while i < len(comp_str):
        if i + 1 < len(comp_str) and comp_str[i:i+2] in elements:
            composition[comp_str[i:i+2]] += 1
            i += 2
        elif comp_str[i:i+1] in elements:
            composition[comp_str[i:i+1]] += 1
            i += 1
        else:
            i += 1
    
    total = sum(composition.values())
    if total > 0:
        composition_vector = np.array([composition[elem] / total for elem in elements])
    else:
        composition_vector = np.zeros(len(elements))
    
    return composition_vector


def load_all_data(base_path):
    base_path = Path(base_path)
    
    if not base_path.exists():
        raise FileNotFoundError(f"Data path not found: {base_path}")
    
    dataset = []
    composition_folders = [d for d in base_path.iterdir() if d.is_dir()]
    
    print(f"Loading {len(composition_folders)} compositions...")
    
    for comp_folder in sorted(composition_folders):
        comp_name = comp_folder.name
        runs_data = load_composition_data_with_geometry(base_path, comp_name)
        
        if not runs_data:
            continue
        
        lq_features, avg_E_S, avg_E_H, descriptor_D = compute_lq_features(runs_data, comp_name)
        
        dataset.append({
            'composition': comp_name,
            'lq_features': lq_features,
            'E_S': avg_E_S,
            'E_H': avg_E_H,
            'descriptor_D': descriptor_D,
            'runs_data': runs_data
        })
    
    return dataset


dataset = load_all_data(ACTIVE_PATH)


In [ ]:
class LocalQuadraticModel:    
    def __init__(self, degree=2, alpha=1.0):

        self.degree = degree
        self.alpha = alpha
        self.poly = PolynomialFeatures(degree=degree, include_bias=True)
        self.scaler = StandardScaler()
        self.model = Ridge(alpha=alpha)
        self.fitted = False
    
    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        
        X_poly = self.poly.fit_transform(X_scaled)
        
        self.model.fit(X_poly, y)
        self.fitted = True
        
        return self
    
    def predict(self, X):

        if not self.fitted:
            raise RuntimeError("Model must be fitted before prediction")
        
        X_scaled = self.scaler.transform(X)
        X_poly = self.poly.transform(X_scaled)
        
        return self.model.predict(X_poly)


In [ ]:
class HEA_GNN(nn.Module):

    
    def __init__(self, input_dim, hidden_dim=64, num_layers=3, dropout=0.2):

        super(HEA_GNN, self).__init__()
        
        self.num_layers = num_layers
        self.dropout = dropout
        
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(input_dim, hidden_dim))
        
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
        
        self.fc1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc2 = nn.Linear(hidden_dim // 2, 1)
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = global_mean_pool(x, batch)
        
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        out = self.fc2(x)
        
        return out.squeeze()


def create_graph_from_structure(atoms, positions, target_energy, element_to_idx):
    adsorbates = ['S', 'H']
    metal_indices = [i for i, atom in enumerate(atoms) if atom not in adsorbates]
    
    if len(metal_indices) == 0:
        return None
    
    metal_atoms = [atoms[i] for i in metal_indices]
    metal_positions = positions[metal_indices]
    
    num_elements = len(element_to_idx)
    node_features = np.zeros((len(metal_atoms), num_elements))
    
    for i, atom in enumerate(metal_atoms):
        if atom in element_to_idx:
            node_features[i, element_to_idx[atom]] = 1.0
    
    cutoff = 3.5
    edge_list = []
    
    for i in range(len(metal_positions)):
        for j in range(i + 1, len(metal_positions)):
            dist = np.linalg.norm(metal_positions[i] - metal_positions[j])
            if dist < cutoff:
                edge_list.append([i, j])
                edge_list.append([j, i])
    
    if len(edge_list) == 0:
        edge_list = [[i, i] for i in range(len(metal_atoms))]
    
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    x = torch.tensor(node_features, dtype=torch.float)
    y = torch.tensor([target_energy], dtype=torch.float)
    
    graph = Data(x=x, edge_index=edge_index, y=y)
    
    return graph


def prepare_gnn_dataset(dataset):

    all_elements = set()
    adsorbates = ['S', 'H']
    
    for comp_data in dataset:
        for run in comp_data['runs_data']:
            for atom in run['atoms']:
                if atom not in adsorbates:
                    all_elements.add(atom)
    
    element_to_idx = {elem: idx for idx, elem in enumerate(sorted(all_elements))}
    
    graph_list = []
    
    for comp_data in dataset:
        target = comp_data['descriptor_D']
        
        if comp_data['runs_data']:
            run = comp_data['runs_data'][0]
            graph = create_graph_from_structure(
                run['atoms'], 
                run['positions'], 
                target, 
                element_to_idx
            )
            
            if graph is not None:
                graph_list.append(graph)
    
    return graph_list, element_to_idx


def train_gnn(model, train_loader, optimizer, device):

    model.train()
    total_loss = 0
    
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        
        out = model(data)
        loss = F.mse_loss(out, data.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * data.num_graphs
    
    return total_loss / len(train_loader.dataset)


def evaluate_gnn(model, loader, device):

    model.eval()
    predictions = []
    targets = []
    
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)
            
            predictions.extend(out.cpu().numpy())
            targets.extend(data.y.cpu().numpy())
    
    return np.array(predictions), np.array(targets)


In [ ]:
def compute_metrics(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    tau, _ = kendalltau(y_true, y_pred)
    
    return {'MAE': mae, 'R2': r2, 'tau': tau}


def learning_curve_experiment(dataset, train_sizes, n_splits=5, n_epochs_gnn=50):
    
    X_lq = np.array([d['lq_features'] for d in dataset])
    y = np.array([d['descriptor_D'] for d in dataset])
    
    graph_list, element_to_idx = prepare_gnn_dataset(dataset)

    results = {
        'train_sizes': train_sizes,
        'lq': {metric: {size: [] for size in train_sizes} for metric in ['MAE', 'R2', 'tau']},
        'gnn': {metric: {size: [] for size in train_sizes} for metric in ['MAE', 'R2', 'tau']}
    }
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    for train_size in train_sizes:
        
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        
        for fold, (train_idx, test_idx) in enumerate(kf.split(X_lq), 1):
            print(f"\nFold {fold}/{n_splits}")
            
            if len(train_idx) > train_size:
                train_idx = np.random.choice(train_idx, train_size, replace=False)
            
            X_train_lq = X_lq[train_idx]
            X_test_lq = X_lq[test_idx]
            y_train = y[train_idx]
            y_test = y[test_idx]
            
            lq_model = LocalQuadraticModel(degree=2, alpha=1.0)
            lq_model.fit(X_train_lq, y_train)
            y_pred_lq = lq_model.predict(X_test_lq)
            
            lq_metrics = compute_metrics(y_test, y_pred_lq)
            
            for metric_name, value in lq_metrics.items():
                results['lq'][metric_name][train_size].append(value)
            
            print(f"  LQ - MAE: {lq_metrics['MAE']:.4f}, R²: {lq_metrics['R2']:.4f}, τ: {lq_metrics['tau']:.4f}")
            
            train_graphs = [graph_list[i] for i in train_idx]
            test_graphs = [graph_list[i] for i in test_idx]
            
            train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
            test_loader = DataLoader(test_graphs, batch_size=32, shuffle=False)
            
            input_dim = len(element_to_idx)
            gnn_model = HEA_GNN(input_dim=input_dim, hidden_dim=64, num_layers=3).to(device)
            optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.001, weight_decay=1e-5)
            
            for epoch in range(n_epochs_gnn):
                train_loss = train_gnn(gnn_model, train_loader, optimizer, device)
            
            y_pred_gnn, y_test_gnn = evaluate_gnn(gnn_model, test_loader, device)
            gnn_metrics = compute_metrics(y_test_gnn, y_pred_gnn)
            
            for metric_name, value in gnn_metrics.items():
                results['gnn'][metric_name][train_size].append(value)
            
            print(f"  GNN - MAE: {gnn_metrics['MAE']:.4f}, R²: {gnn_metrics['R2']:.4f}, τ: {gnn_metrics['tau']:.4f}")
    
    return results


In [ ]:
train_sizes = [80, 120, 160, 200, 240, 280, 320, 360, 400]

max_train_size = int(len(dataset) * 0.8)
train_sizes = [size for size in train_sizes if size <= max_train_size]


results = learning_curve_experiment(
    dataset=dataset,
    train_sizes=train_sizes,
    n_splits=5,
    n_epochs_gnn=50
)


In [ ]:
def plot_learning_curves(results):

    train_sizes = results['train_sizes']
    metrics = ['MAE', 'R2', 'tau']
    metric_labels = {
        'MAE': 'MAE vs Training Size',
        'R2': 'R² vs Training Size',
        'tau': 'τ vs Training Size'
    }
    
    max_size = max(train_sizes)
    if max_size <= 200:
        system_label = "9-Metal System"
    else:
        system_label = "19-Metal System"
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Learning Curves: {system_label}', fontsize=14, fontweight='bold')
    
    colors = {
        'lq': '#1f77b4',  
        'gnn': '#2ca02c'  
    }
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        
        for model_name in ['lq', 'gnn']:
            means = []
            stds = []
            
            for size in train_sizes:
                values = results[model_name][metric][size]
                means.append(np.mean(values))
                stds.append(np.std(values))
            
            means = np.array(means)
            stds = np.array(stds)
            
            label = 'LQ' if model_name == 'lq' else 'GNN'
            ax.plot(train_sizes, means, '-o', label=label, 
                   color=colors[model_name], linewidth=2, markersize=6)
            ax.fill_between(train_sizes, means - stds, means + stds, 
                           alpha=0.2, color=colors[model_name])
        
        ax.set_xlabel('Training Set Size', fontsize=11)
        ax.set_ylabel(metric, fontsize=11)
        ax.set_title(metric_labels[metric], fontsize=12)
        ax.legend(loc='best', frameon=True, shadow=True)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
     
    largest_size = train_sizes[-1]
    
    for model_name in ['lq', 'gnn']:
        print(f"\n{model_name.upper()} Model (n={largest_size}):")
        for metric in metrics:
            values = results[model_name][metric][largest_size]
            mean_val = np.mean(values)
            std_val = np.std(values)
            print(f"  {metric:4s}: {mean_val:.4f} ± {std_val:.4f}")
plot_learning_curves(results)


In [ ]:
def export_results(results, filename='learning_curve_results.csv'):

    records = []
    
    for train_size in results['train_sizes']:
        for model_name in ['lq', 'gnn']:
            for metric in ['MAE', 'R2', 'tau']:
                values = results[model_name][metric][train_size]
                
                records.append({
                    'training_size': train_size,
                    'model': model_name.upper(),
                    'metric': metric,
                    'mean': np.mean(values),
                    'std': np.std(values),
                    'min': np.min(values),
                    'max': np.max(values)
                })
    
    df = pd.DataFrame(records)
    df.to_csv(filename, index=False)
    
    print(f"\n✓ Results exported to {filename}")
    print(f"  Total records: {len(records)}")
    
    return df


results_df = export_results(results)
print("\nPreview of exported data:")
print(results_df.head(10))
